# GenMol: Interactive Molecule Generation

**GenMol** is a discrete diffusion model for drug-like molecule generation, developed by NVIDIA. This notebook lets you run the model interactively.

> You can run the model and play around with the parameters in this notebook. **To edit code**, please click on "Copy to Drive" to save your own version of the notebook to edit.

> **Before you start:** go to *Runtime → Change runtime type* and select **T4 GPU**. Generation will work on CPU too, but will be significantly slower.

---

## Setup

Run the cell below once. It clones the repository, installs dependencies, and downloads the model checkpoint. This takes roughly 3–5 minutes. In the meantime, read below how the model works!

In [ ]:
import os, site, sys, warnings

print("Cloning repository...")
if not os.path.exists("genmol"):
    !git clone -q https://github.com/mlederbauer/genmol.git
os.chdir("genmol")

# Pin versions compatible with Colab
for f in ["env/requirements.txt", "pyproject.toml"]:
    !sed -i '/numpy/d' {f}
    !sed -i 's/pandas==2.1.0/pandas/g' {f}
    !sed -i 's/wandb==0.13.5/wandb/g' {f}
    !sed -i '/torch==/d' {f}
    !sed -i '/pytdc/d' {f}

print("Installing dependencies (~2 min)...")
!pip install -q "numpy>=2.0.0,<2.1.0"
!pip install -q -r env/requirements.txt
!pip install -q "setuptools<78" scikit-learn plotly rdkit ipywidgets
!pip install -q fuzzywuzzy pytdc==0.4.1 --no-deps
!pip install -q -e . --no-deps
!pip install -q -U "numpy>=2.0.0,<2.1.0" --force-reinstall

# RDKit pandas patch
try:
    p = os.path.join(site.getsitepackages()[0], 'rdkit', 'Chem', 'PandasPatcher.py')
    if os.path.exists(p):
        txt = open(p).read().replace(
            'pandas_formats.format, get_adjustment_name',
            'get_adjustment_module, get_adjustment_name'
        )
        open(p, 'w').write(txt)
except Exception:
    pass

print("Downloading model checkpoint (~1.6 GB)...")
os.makedirs("checkpoints", exist_ok=True)
!wget -q --show-progress -O checkpoints/model_v2.ckpt \
    https://huggingface.co/nvidia/NV-GenMol-89M-v2/resolve/main/model_v2.ckpt

warnings.filterwarnings("ignore")
sys.path.insert(0, "src")

In [ ]:
from genmol.sampler import Sampler
import genmol.app as app

sampler = Sampler("checkpoints/model_v2.ckpt")

print("Downloading PubChem sample...")
!wget -q --show-progress \
    https://ftp.ncbi.nlm.nih.gov/pubchem/Compound/Extras/CID-SMILES.gz \
    -O CID-SMILES.gz
!gunzip -f CID-SMILES.gz

print("Ready.")


### How the model works

GenMol represents molecules as sequences of **fragments** — the ring systems and chains that chemists already think in. Generation is then a fill-in-the-blank problem: some fragments are fixed as context, and the model fills in the rest. The mechanism is called **masked diffusion**: during training, fragments are progressively replaced with a `[MASK]` placeholder until the whole molecule is hidden. The model learns to reverse this, recovering the original structure. At generation time, it starts from a fully masked sequence and iteratively fills in fragments until a complete molecule emerges.

This framing related to three tasks in fragment-based drug discovery. In ***de novo* generation**, nothing is fixed and an entire new molecule is generated. In **scaffold decoration**, the user sets the core ring system and the model generates side chains. For **linker design**, the user provides two terminal fragments and the model generates a bridge between them. In this notebook, you can explore these capabilities yourself!

### Output properties

Each generated molecule is annotated with computed physicochemical properties:

- **QED** (Quantitative Estimate of Drug-likeness, 0–1): a composite score capturing drug-likeness based on molecular weight, lipophilicity, hydrogen bonding, and other properties. Higher is more drug-like.
- **MW**: molecular weight in Da
- **LogP**: estimated octanol–water partition coefficient (lipophilicity)
- **HBD / HBA**: hydrogen bond donors and acceptors

### Generation parameters

Two parameters are worth exploring:

- **Temperature**: lower values keep sampling close to common, drug-like structures; higher values push the model toward less typical regions of chemical space.
- **Randomness**: controls which fragments get unmasked first, adding a second source of variation on top of temperature.
- **Gamma**, which is available for scaffold decoration and linker design: controls how strongly the model is guided by the provided fragment.

Happy generating!

---
## De Novo Generation

The model generates molecules from scratch, with no structural constraints. Adjust the sliders and click **Generate**.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
import genmol.app as app

n_samples  = widgets.IntSlider(value=12, min=1, max=50, description="Samples:", style={"description_width": "80px"})
temp       = widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.1, description="Temperature:", style={"description_width": "80px"})
randomness = widgets.FloatSlider(value=0.3, min=0.0, max=5.0, step=0.1, description="Randomness:", style={"description_width": "80px"})
btn        = widgets.Button(description="Generate", button_style="primary")
out        = widgets.Output()

def run(_):
    with out:
        clear_output()
        print("Generating...")
        results = sampler.de_novo_generation(
            num_samples=n_samples.value,
            softmax_temp=temp.value,
            randomness=randomness.value,
        )
        app.generated_smiles = results
        app.display_results(results)

btn.on_click(run)
display(widgets.VBox([
    widgets.HBox([n_samples, temp, randomness]),
    btn, out
]))

---
## Scaffold Decoration

Fix a core scaffold and let the model add side chains. Draw your scaffold or type a SMILES directly — no attachment points needed, the model finds them automatically.

**Examples:** `c1ccc2ncccc2c1` (quinoline), `c1ccc(cc1)C(=O)N` (benzamide)

In [ ]:
from IPython.display import HTML, display
from google.colab import output
import genmol.app as app

scaffold_smiles = ""

def _save_scaffold(smiles):
    global scaffold_smiles
    scaffold_smiles = app.clean_fragment(smiles)
    print(f"Scaffold saved: {scaffold_smiles}")

output.register_callback('notebook.set_scaffold', _save_scaffold)

display(HTML("""
<h4>Draw scaffold</h4>
<script src="https://jsme-editor.github.io/dist/jsme/jsme.nocache.js"></script>
<div id="jsme_scaf"></div>
<button onclick="google.colab.kernel.invokeFunction('notebook.set_scaffold',[jsmeScaf.smiles()],{});"
  style="margin-top:8px;padding:8px 16px;background:#1a73e8;color:white;border:none;border-radius:4px;cursor:pointer">
  Save scaffold
</button>
<script>function jsmeOnLoad(){jsmeScaf=new JSApplet.JSME("jsme_scaf","380px","280px",{options:"query,react,multipart,star"});}</script>
"""))

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
import genmol.app as app

smiles_input = widgets.Text(value="", placeholder="or type SMILES here, e.g. c1ccc2ncccc2c1",
                             description="SMILES:", layout=widgets.Layout(width="420px"),
                             style={"description_width": "60px"})
n_samples  = widgets.IntSlider(value=12, min=1, max=50, description="Samples:", style={"description_width": "80px"})
temp       = widgets.FloatSlider(value=1.5, min=0.5, max=2.0, step=0.1, description="Temperature:", style={"description_width": "80px"})
randomness = widgets.FloatSlider(value=2.0, min=0.0, max=5.0, step=0.1, description="Randomness:", style={"description_width": "80px"})
gamma      = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05, description="Gamma:", style={"description_width": "80px"})
btn        = widgets.Button(description="Decorate", button_style="primary")
out        = widgets.Output()

def run(_):
    global scaffold_smiles
    with out:
        clear_output()
        smi = smiles_input.value.strip() or scaffold_smiles
        if not smi:
            print("Please draw a scaffold above or type a SMILES.")
            return
        print(f"Decorating {smi}...")
        results = sampler.fragment_completion(
            fragment=smi,
            num_samples=n_samples.value,
            apply_filter=True,
            softmax_temp=temp.value,
            randomness=randomness.value,
            gamma=gamma.value,
        )
        app.generated_smiles = results
        app.display_results(results)

btn.on_click(run)
display(widgets.VBox([
    smiles_input,
    widgets.HBox([n_samples, temp, randomness, gamma]),
    btn, out
]))

---
## Linker Design

Provide two fragments; the model generates a linker that connects them. Each fragment **must** have an attachment point marked as `[*]` (hover over the atom you want to act as a linker in the editor and type "R", or type `[*]` directly in the SMILES).

**Examples:** Fragment 1: `[*]c1ccc2c(c1)cccc2`, Fragment 2: `[*]CCO`

In [ ]:
from IPython.display import HTML, display
from google.colab import output
import genmol.app as app

frag1_smiles = ""
frag2_smiles = ""

def _save_linker(s1, s2):
    global frag1_smiles, frag2_smiles
    if not s1 or not s2:
        print("Both canvases must contain a molecule.")
        return
    frag1_smiles = app.clean_fragment(s1)
    frag2_smiles = app.clean_fragment(s2)
    if "[*]" not in frag1_smiles or "[*]" not in frag2_smiles:
        print("Both fragments must have an attachment point [*].")
        return
    print(f"Fragments saved: {frag1_smiles}  |  {frag2_smiles}")

output.register_callback('notebook.set_linker', _save_linker)

display(HTML("""
<h4>Draw fragments</h4>
<script src="https://jsme-editor.github.io/dist/jsme/jsme.nocache.js"></script>
<div style="display:flex;gap:24px">
  <div><p><b>Fragment 1</b></p><div id="jsme_f1"></div></div>
  <div><p><b>Fragment 2</b></p><div id="jsme_f2"></div></div>
</div>
<button onclick="google.colab.kernel.invokeFunction('notebook.set_linker',[jsmeF1.smiles(),jsmeF2.smiles()],{});"
  style="margin-top:8px;padding:8px 16px;background:#1a73e8;color:white;border:none;border-radius:4px;cursor:pointer">
  Save both fragments
</button>
<script>
function jsmeOnLoad(){
  jsmeF1=new JSApplet.JSME("jsme_f1","340px","260px",{options:"query,react,multipart,star"});
  jsmeF2=new JSApplet.JSME("jsme_f2","340px","260px",{options:"query,react,multipart,star"});
}
</script>
"""))

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
import genmol.app as app

f1_input   = widgets.Text(value="", placeholder="[*]c1ccc2c(c1)cccc2",
                           description="Fragment 1:", layout=widgets.Layout(width="380px"),
                           style={"description_width": "80px"})
f2_input   = widgets.Text(value="", placeholder="[*]CCO",
                           description="Fragment 2:", layout=widgets.Layout(width="380px"),
                           style={"description_width": "80px"})
n_samples  = widgets.IntSlider(value=12, min=1, max=50, description="Samples:", style={"description_width": "80px"})
temp       = widgets.FloatSlider(value=1.2, min=0.5, max=2.0, step=0.1, description="Temperature:", style={"description_width": "80px"})
randomness = widgets.FloatSlider(value=3.0, min=0.0, max=5.0, step=0.1, description="Randomness:", style={"description_width": "80px"})
gamma      = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05, description="Gamma:", style={"description_width": "80px"})
btn        = widgets.Button(description="Design linker", button_style="primary")
out        = widgets.Output()

def run(_):
    global frag1_smiles, frag2_smiles
    with out:
        clear_output()
        s1 = f1_input.value.strip() or frag1_smiles
        s2 = f2_input.value.strip() or frag2_smiles
        if not s1 or not s2:
            print("Please provide both fragments.")
            return
        if "[*]" not in s1 or "[*]" not in s2:
            print("Both fragments must contain [*] as an attachment point.")
            return
        fragment = f"{s1}.{s2}"
        print(f"Linking {fragment}...")
        results = sampler.fragment_linking_onestep(
            fragment=fragment,
            num_samples=n_samples.value,
            softmax_temp=temp.value,
            randomness=randomness.value,
            gamma=gamma.value,
        )
        app.generated_smiles = results
        app.display_results(results)

btn.on_click(run)
display(widgets.VBox([
    widgets.HBox([f1_input, f2_input]),
    widgets.HBox([n_samples, temp, randomness, gamma]),
    btn, out
]))

---
## Chemical Space Visualisation

Where do the generated molecules sit in chemical space relative to known compounds?

The cell below loads a sample of PubChem molecules, computes ECFP4 fingerprints for both the reference set and your generated molecules, and projects everything onto a 2D plane using PCA. Grey dots are PubChem reference compounds; coloured markers are your generated molecules, coloured by QED score. Hover over a generated molecule to see its structure and properties.

Run a generation task above first, then run this cell.

In [ ]:
# Load a PubChem sample for the reference background
import pandas as pd

REF_FILE = "pubchem_sample.smi"
N_REF    = 10000000 # number of reference molecules to use

if not os.path.exists(REF_FILE):
    # Take first N_REF entries and save as a plain SMILES file
    df_ref = pd.read_csv("CID-SMILES", sep="\t", header=None,
                         names=["cid", "smiles"], nrows=N_REF)
    df_ref["smiles"].to_csv(REF_FILE, index=False, header=False)
    print(f"Saved {N_REF} reference SMILES to {REF_FILE}")

reference_smiles = pd.read_csv(REF_FILE, header=None)[0].dropna().tolist()
print(f"Reference set: {len(reference_smiles)} molecules")

In [ ]:
import genmol.app as app

app.plot_chemical_space(reference_smiles, app.generated_smiles)